# Customer Churn Analysis

In this notebook, we analyse the products that lead to customer churn. By calculating the standard deviation of intervals between two purchases of each customer plus, we define the churn threshold as the standard deviation plus the mean of intervals. After identifying the churned customers, we can find out the products that are likely to lead to customer churn.

Then we analyse the products that drive customer retention, and we will compare the retention rate and churn rate of a product.  

In [ ]:
# read bigquery data into pandas dataframe
import pandas as pd
import numpy as np
import pandas_gbq
import matplotlib.pyplot as plt
from collections import Counter
from itertools import combinations
#pd.set_option('display.max_colwidth', None)
df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.cafe.cafe-sales`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)
df.dropna(inplace=True)
df.drop(df[df['status'] != 2].index, inplace=True)

In [ ]:
item_details = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.dbt_cafeanalytics.item_details`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)
item_details.dropna(inplace=True)

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
df

In [ ]:
print(item_details[item_details['item_quantity']!=1])

In [ ]:
merged_data = pd.merge(df, item_details, on='order_id')
# calculate the total revenue of each item in a single order
merged_data['total_item_sales'] = merged_data['item_quantity'] * merged_data['item_price']
# calculate the total revenue of each item in this period
product_sales = merged_data.groupby('item_name').agg(
    total_sales=('total_item_sales', 'sum')
).reset_index()
total_sales_all = product_sales['total_sales'].sum()
#calculate the market share of each product
product_sales['market_share'] = product_sales['total_sales'] / total_sales_all
# calculate growth rate of each product
merged_data['year'] = pd.to_datetime(merged_data['date_paid']).dt.year
annual_sales = merged_data.groupby(['year', 'item_name'])['total_item_sales'].sum().reset_index()


growth_rate = annual_sales.groupby('item_name').agg(
    growth_rate=('total_item_sales', lambda x: (x.iloc[-1] - x.iloc[0]) / x.iloc[0] if x.size > 1 else 0)
).reset_index()
# define threshold of market share and growth rate to identify the products with low growth and low market share
final_data = pd.merge(product_sales, growth_rate, on='item_name')
threshold_share = product_sales['market_share'].quantile(0.3) 
threshold_growth = 0.0
low_share_low_growth = final_data[
    (final_data['market_share'] < threshold_share) & 
    (final_data['growth_rate'] < threshold_growth)
]
print(low_share_low_growth)

## find products that lead to customer churn and products that drive customer retention

By calculating the standard deviation of intervals between two purchases of each customer plus, we define the churn threshold as the standard deviation plus the mean of intervals.For a single purchase, if the interval is greater than the churn threshold, than we mark the last purchase as churn.Then we identify the products the loss of which caused because of churning customers is greater than the benefits brought by retaining customers.To quantify the revenue generated by each customer, we use the customer's LTV as a standard measure for evaluation.

In [ ]:
# the following code is used to label each purchase as 'churn' or 'not churn'
df=df.sort_values(by=['customer_id','date_paid'])
# define interval as the days between two purchases
df['interval'] = df.groupby('customer_id')['date_paid'].diff().dt.days
intervals = df.groupby('customer_id')['interval'].apply(lambda x:x.dropna())
# define churn thresholds as the mean of intervals plus the standard deviation of intervals for each customer
churn_thresholds = intervals.groupby(level=0).agg(['mean','std']).reset_index()
churn_thresholds['churn_threshold'] =  churn_thresholds['std'] + churn_thresholds['mean']
df['is_churned'] = False
# if interval is greater the threshold, then label the customer as churn
for customer_id, group in df.groupby('customer_id'):
    for i in range(1, len(group)):     
        threshold_row = churn_thresholds[churn_thresholds['customer_id'] == customer_id]
        if not threshold_row.empty:
            churn_threshold = threshold_row['churn_threshold'].values[0]
            if group['interval'].iloc[i] > churn_threshold:
                df.loc[group.index[i-1], 'is_churned'] = True
single_appearance = df['customer_id'].value_counts() == 1
single_customers = single_appearance[single_appearance].index
df.loc[df['customer_id'].isin(single_customers), 'is_churned'] = True

multiple_customers = df['customer_id'].value_counts()[df['customer_id'].value_counts() > 1].index

# Global maximum date_paid
max_date_paid = df['date_paid'].max()

# For each customer_id that appears multiple times, calculate its maximum date_paid
for customer_id in multiple_customers:
    customer_max_date_paid = df.loc[df['customer_id'] == customer_id, 'date_paid'].max()
    
    # Calculate the time difference
    time_difference = (max_date_paid - customer_max_date_paid).days
    
    # Get the corresponding churn_threshold
    threshold_value = churn_thresholds.loc[churn_thresholds['customer_id'] == customer_id, 'churn_threshold'].values[0]
    
    # If the time difference is greater than the threshold, mark as True
    if time_difference > threshold_value:
        df.loc[df['customer_id'] == customer_id, 'is_churned'] = True

In [ ]:
result = df[['customer_id','interval','is_churned']].iloc[-500:]  # 这将返回最后 100 行
print(result)

## find the products that are likely to lead to customer churn by calculating the churn rate and retention rate

In [ ]:
# find the products that are likely to lead to customer churn by calculating the churn rate and retention rate
churn_orders = df[df['is_churned']].copy()
churned_order_ids = churn_orders['order_id'].unique()
churned_orders_details = item_details[item_details['order_id'].isin(churned_order_ids)]
# present number of customers churn each product leads to
product_churn_analysis = churned_orders_details['item_name'].value_counts().reset_index()
product_churn_analysis.columns=['item_name','churned_customer_count']
# present sale count of each product that lead to customer churn
product_count_churn_analysis = churned_orders_details.groupby('item_name')['order_id'].count().reset_index()
product_count_churn_analysis.columns=['item_name','sale_churn_count']
#print(product_count_churn_analysis)

# find the products that drive customer retention
retention_orders = df[~df['is_churned']].copy()
retention_order_ids = retention_orders['order_id'].unique()
retention_order_details = item_details[item_details['order_id'].isin(retention_order_ids)]
# present number of customers retention each product leads to
product_retention_analysis = retention_order_details['item_name'].value_counts().reset_index()
product_retention_analysis.columns=['item_name','retained_customer_count']

# present sale count of each product that lead to customer retention
product_count_retention_analysis = retention_order_details.groupby('item_name')['order_id'].count().reset_index()
product_count_retention_analysis.columns=['item_name','sale_retention_count']
#print(product_count_retention_analysis)
# compare customer retention and customer churn of each product
product_performance = pd.merge(product_churn_analysis, product_retention_analysis, on='item_name', how = 'outer')
product_performance['total_churned'] = product_performance['churned_customer_count'].fillna(0)
product_performance['total_retained'] = product_performance['retained_customer_count'].fillna(0)
product_performance['churn_rate'] = product_performance['total_churned'] / (product_performance['total_churned'] + product_performance['total_retained'])
product_performance['retention_rate'] = product_performance['total_retained'] / (product_performance['total_churned'] + product_performance['total_retained'])
product_performance = product_performance.merge(product_count_retention_analysis,on='item_name', how='outer')
product_performance = product_performance.merge(product_count_churn_analysis,on='item_name', how='outer')
product_performance.fillna(0,inplace=True)
print(product_performance[['item_name', 'total_churned', 'total_retained', 'churn_rate', 'retention_rate']])

In [ ]:
df

In [ ]:

start_date = '2022-07-01'
end_date = '2023-07-01'


#loyal_customers = df[
 #   (df['date_paid'] >= start_date) & 
  #  (df['date_paid'] <= end_date)
#].groupby('customer_id').filter(lambda x: (x['is_churned'] == False).all())['customer_id'].unique()
# customers whose retention counts is greater than churn counts are defined as loyal customers 
loyal_period = (df['date_paid'] >= '2022-07-01') & (df['date_paid'] <= '2023-07-01')
post_loyal_period = df['date_paid'] > '2023-07-01'


loyal_counts = df[loyal_period].groupby('customer_id')['is_churned'].value_counts().unstack(fill_value=0)
post_loyal_counts = df[post_loyal_period].groupby('customer_id')['is_churned'].value_counts().unstack(fill_value=0)

# identify loyal customers
loyal_customers = loyal_counts[loyal_counts[False] > loyal_counts[True]].index

# identify lost customers
lost_customers = post_loyal_counts[post_loyal_counts[False] < post_loyal_counts[True]].index
print(lost_customers)

In [ ]:
def simulate_recovery(df, high_risk_products):
    # delete high risk products
    filtered_df = df[~df['item_name'].isin(high_risk_products['item_name'])]

    # calculate interval
    filtered_df['date_paid'] = pd.to_datetime(filtered_df['date_paid'])
    filtered_df['interval'] = filtered_df.groupby('customer_id')['date_paid'].diff().dt.days
    
    # calculate the threshold again
    customer_intervals = filtered_df.groupby('customer_id')['interval'].agg(['mean', 'std']).reset_index()
    customer_intervals['churn_threshold'] = customer_intervals['mean'] + customer_intervals['std']
    filtered_df = filtered_df.merge(customer_intervals[['customer_id', 'churn_threshold']], on='customer_id', how='left')
    # deal with single purchase
    single_purchase_customers = filtered_df['customer_id'].value_counts()
    single_purchase_customers = single_purchase_customers[single_purchase_customers == 1].index
    
    filtered_df['is_churned'] = False


    for idx in range(1, len(filtered_df)):
        if (filtered_df['interval'].iloc[idx] > churn_threshold) and (filtered_df['customer_id'].iloc[idx] == filtered_df['customer_id'].iloc[idx - 1]):
            filtered_df.at[idx - 1, 'is_churned'] = True
    filtered_df.loc[filtered_df['customer_id'].isin(single_purchase_customers), 'is_churned'] = True
    
    
    post_loyal_customers_filtered = filtered_df[
        (filtered_df['customer_id'].isin(loyal_customers)) & 
        (filtered_df['date_paid'] > '2023-07-01')
    ]
    
    post_loyal_customers_filtered = post_loyal_customers_filtered.sort_values('date_paid').groupby('customer_id').first().reset_index()
    lost_customers_filtered = post_loyal_customers_filtered[post_loyal_customers_filtered['is_churned'] == True]
    
    original_loyal_customers = df[
        (df['customer_id'].isin(loyal_customers)) & 
        (df['date_paid'] > '2023-07-01')
    ]
    
    post_loyal_customers = original_loyal_customers.sort_values('date_paid').groupby('customer_id').first().reset_index()
    lost_customers = post_loyal_customers[post_loyal_customers['is_churned'] == True]
    print(lost_customers)
    loyal_period = (filtered_df['date_paid'] >= '2022-07-01') & (filtered_df['date_paid'] <= '2023-07-01')
    post_loyal_period = filtered_df['date_paid'] > '2023-07-01'
    
    
   # lost_customer_ids = post_loyal_customers_filtered[post_loyal_customers_filtered['is_churned'] == True]['customer_id'].unique()
    recovered_customer_ids = len(lost_customers) - len(lost_customers_filtered)
    a=len(lost_customers)
    b=len(lost_customers_filtered)
    # calculate the number of customers that would not be chused with high risk products deleted
    recovered_count = recovered_customer_ids
    
    return recovered_count

# identify high risk products
product_performance['risk_score'] = product_performance['churn_rate'] / product_performance['retention_rate']
item_nonact = product_performance[product_performance['sale_retention_count']+product_performance['sale_churn_count']<30]['item_name']
filtered_high_risk_products = product_performance[~product_performance['item_name'].isin(item_nonact)]


top_risk_products = filtered_high_risk_products.nlargest(10, 'risk_score')
print(top_risk_products)
merged_purchase = pd.merge(df, item_details, on='order_id')


recovery_5 = simulate_recovery(merged_purchase, top_risk_products.head(10))
#recovery_10 = simulate_recovery(merged_purchase, high_risk_products)

print(recovery_5)


## compare the revenue and loss each product brings with life time value of customers taken into consideration

In [ ]:
merged_df = df.merge(item_details, on='order_id', how='inner')

In [ ]:
merged_df['revenue'] = merged_df['item_quantity'] * merged_df['item_price']
# calculate the total revenue of each customer
customer_ltv = merged_df.groupby('customer_id')['revenue'].sum().reset_index()
customer_ltv.columns = ['customer_id', 'total_revenue']
ltv_df = merged_df.merge(customer_ltv, on='customer_id', how='left')

# Calculate revenue from churned customers
churned_revenue_summary = ltv_df[ltv_df['is_churned'] == True].groupby('item_name')['total_revenue'].sum().reset_index()
churned_revenue_summary.columns = ['item_name', 'total_revenue_churned']

# Calculate revenue from retained customers
retained_revenue_summary = ltv_df[ltv_df['is_churned'] == False].groupby('item_name')['total_revenue'].sum().reset_index()
retained_revenue_summary.columns = ['item_name', 'total_revenue_retained']
result_ltv = pd.merge(churned_revenue_summary, retained_revenue_summary, on='item_name', how='outer').fillna(0)
# identify items that cause loss because of the negative influence of churned customers greater than benefits of retaining customers
result_ltv['revenue_minus_loss'] = result_ltv['total_revenue_retained'] - result_ltv['total_revenue_churned']
print(result_ltv[result_ltv['revenue_minus_loss']<0])

### products that should be considered for deletion

products should be considered for deletion if the total loss caused by churning customers is greater than 2 times the total revenue because of retaining customers

In [ ]:
item_nonactive = product_performance[product_performance['sale_retention_count']+product_performance['sale_churn_count']<30]['item_name']
deletion_products = result_ltv[
(result_ltv['total_revenue_churned'] > 2 * result_ltv['total_revenue_retained']) &
(~result_ltv['item_name'].isin(item_nonactive))                                                                 
][['item_name','revenue_minus_loss']]
print(deletion_products)

### products under consideration for upgrade

Products should be considered for upgrade if the total loss caused by churning customers is greater than revenue by retaining customers but less than 1.2 times the total revenue by retaining customers. 

In [ ]:
#delete_threshold = 0.05  # threshold of churn rate
# max_sales_threshold = 10 # threshold of number of churning customers 

#items_to_exclude = product_performance[product_performance['sale_retention_count']>50]['item_name']
loss_products = result_ltv[result_ltv['revenue_minus_loss']<0]['item_name']
#item_nonactive = product_performance[product_performance['sale_retention_count']+product_performance['sale_churn_count']<30]['item_name']
pet_products = low_share_low_growth['item_name']
# items that may be considered to be upgraded
products_less_popular = product_performance[
    (product_performance['item_name'].isin(loss_products)) &
    (~product_performance['item_name'].isin(deletion_products)) &
    (~product_performance['item_name'].isin(item_nonactive))
]
products_for_upgrade = pd.merge(products_less_popular, result_ltv, on='item_name', how='inner')
#products_to_delete = pd.merge(products_less_popular, product_sales, on='item_name', how='left')
#products_to_delete_df = pd.merge(products_to_delete, growth_rate, on='item_name', how='left')
print(products_for_upgrade[['item_name','revenue_minus_loss']])

### products to be monitored
if the market share and growth rate is low, and the churn rate is greater than the retention rate, then the products should be monitored. 

In [ ]:
#item_nonactive = product_performance[product_performance['sale_retention_count']+product_performance['sale_churn_count']<30]['item_name']
products_less_popular = product_performance[
    (product_performance['churn_rate'] > product_performance['retention_rate']) &
    #(~product_performance['item_name'].isin(item_nonactive)) &
    (~product_performance['item_name'].isin(deletion_products['item_name'])) &
    (~product_performance['item_name'].isin(products_for_upgrade['item_name'])) &
    (product_performance['item_name'].isin(pet_products))
]
#print(products_less_popular)
products_to_monitor = pd.merge(products_less_popular, product_sales, on='item_name', how='inner')
products_to_monitor_df = pd.merge(products_to_monitor, growth_rate, on='item_name', how='inner')
print(products_to_monitor_df[['item_name','market_share','growth_rate','churn_rate','retention_rate']])
#print(product_performance[~product_performance['item_name'].isin(deletion_products['item_name'])])

In [ ]:
results = []

# Loop through each product in products_to_delete_df
for index, product in products_to_delete_df.iterrows():
    item_name = product['item_name']
    
    # Find orders containing the current product
    orders_with_product = item_details[item_details['item_name'] == item_name]['order_id']
    
    # Get customer spending for these orders
    customers_spending = df[df['order_id'].isin(orders_with_product)]
    
    # Calculate total spending for each customer
    total_spent_per_customer = df.groupby('customer_id')['total'].sum().reset_index()
    total_spent_per_customer.rename(columns={'total': 'total_spent'}, inplace=True)
    
    # Calculate spending on the specific product
    product_spending = customers_spending.groupby('customer_id')['total'].sum().reset_index()
    product_spending.rename(columns={'total': 'product_spent'}, inplace=True)
    
    # Merge to get the ratio
    customer_summary = pd.merge(total_spent_per_customer, product_spending, on='customer_id', how='left')
    customer_summary['product_ratio'] = customer_summary['product_spent'] / customer_summary['total_spent']
    
    # Handle missing values
    customer_summary.dropna(inplace=True)

    # Add product name to the summary
    customer_summary['item_name'] = item_name
    
    # Append results
    results.append(customer_summary)

# Step 3: Concatenate all results into a single DataFrame and rearrange columns
final_summary = pd.concat(results, ignore_index=True)
#final_summary = final_summary[['item_name', 'customer_id', 'total_spent', 'product_spent', 'product_ratio']]

# Output the final summary
print(final_summary[['item_name', 'customer_id', 'total_spent', 'product_ratio']])

In [ ]:
unique_customers = df['customer_id'].nunique()
product_customer = final_summary.groupby('item_name')['customer_id'].size().reset_index()
product_customer.columns=['item_name','number_of_customer']
product_customer['customer_ratio'] = product_customer['number_of_customer'] / unique_customers

average_product_ratio = final_summary.groupby('item_name')['product_ratio'].mean().reset_index()
average_product_ratio.columns = ['item_name', 'average_product_ratio']

product_customer = pd.merge(product_customer, average_product_ratio, on='item_name', how='inner' )
print(product_customer)

In [ ]:
df['date_paid'] = pd.to_datetime(df['date_paid'])
df['year'] = df['date_paid'].dt.year
# calculate the total visits of each custoemr each year
customer_visits = df.groupby(['customer_id', 'year']).size().reset_index(name='visit_count')
customer_total_visits = customer_visits.groupby('customer_id')['visit_count'].sum().reset_index()
one_time_customers = customer_total_visits[customer_total_visits['visit_count'] == 1]
low_frequency_customers = customer_visits[customer_visits['visit_count'] <= 1]['customer_id'].unique()
low_frequency_customers_count = len(low_frequency_customers)
total_customers = df['customer_id'].nunique()
# calculate the ratio of churned customers
churned_count = len(one_time_customers)
churned_ratio = churned_count / total_customers
# calculate the ratio of low frequency customers
low_frequency_count = len(set(low_frequency_customers))
low_frequency_ratio = low_frequency_count / total_customers
print(churned_ratio)